# Hardware Verification

**Purpose:** This notebook strictly serves to verify your trained PyTorch quantum-classical hybrid model using **real IBM Quantum hardware** for inference.

**Constraints & Expectations:**
- 🚫 **No Training:** This notebook runs pure inference. No gradients, no optimizers, no weights updates (`eval()` + `torch.no_grad()`).
- ⚡ **Real Hardware:** Uses Qiskit Provider via PennyLane to submit true executions, not simulated runs.
- 🔐 **Configuration:** Requires an `.env` file with `IBM_QUANTUM_TOKEN` alongside the project root.
- 📁 **Cloud/Drive Storage:** Set up to pull your `model.pt` from a configurable path like Google Drive or local storage.

In [4]:
# Instalar dependencia (si no la tenés ya instalada)
!pip -q install qiskit-ibm-runtime

import os
from pathlib import Path
import torch  # lo dejo porque lo tenías en tu bloque original
from google.colab import userdata
from qiskit_ibm_runtime import QiskitRuntimeService

# 1) Cargar credenciales desde Colab Secrets (nunca hardcodear)
api_key = userdata.get("IBM_QUANTUM_TOKEN")   # o el nombre que hayas usado en Secrets
instance_crn = userdata.get("IBM_CRN")        # opcional, pero recomendado

# 2) Validaciones (estilo similar a tu bloque anterior)
if not api_key:
    raise ValueError(
        "❌ IBM_QUANTUM_TOKEN no encontrado en Colab Secrets.\n"
        "Creá un Secret con ese nombre y habilitá 'Notebook access'."
    )
else:
    print("✅ IBM Quantum API key encontrada.")

if instance_crn:
    print("✅ IBM CRN encontrado.")
else:
    print("⚠️ IBM_CRN no encontrado. Se intentará conectar sin CRN (no recomendado).")

# 3) Inicializar servicio Qiskit Runtime
service = QiskitRuntimeService(
    token=api_key,
    instance=instance_crn  # si es None, Qiskit intenta resolver por defecto
)

print("✅ Conectado OK a Qiskit Runtime.")

qiskit_runtime_service._discover_account:WARNING:2026-02-24 22:48:04,869: Loading account with the given token. A saved account will not be used.


✅ IBM Quantum API key encontrada.
✅ IBM CRN encontrado.
✅ Conectado OK a Qiskit Runtime.


In [5]:
import os, sys, subprocess
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone the repository to make `src` module available
    REPO_URL = 'https://github.com/lburdman/qnn-transfer-learning.git'
    REPO_PATH = Path('/content/qnn-transfer-learning')
    BRANCH = 'main'
    
    if not REPO_PATH.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], check=True)
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'fetch'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
else:
    os.chdir(Path('.'))

sys.path.append(str(Path.cwd() / 'src'))
print('Working dir:', Path.cwd())
print('Python path updated with src/')


## 1. Paths & Configuration
Set your model path, testing dataset path, and the number of shots for the quantum device.

In [6]:
import json
import os
import sys

# --- CONFIGURE THESE PATHS ---
# Example for Colab: "/content/drive/MyDrive/models/my_model.pt"
MODEL_PATH = "/content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/model.pt"
# Small subset for quick inference to avoid huge IBM Q queue times
TEST_DATA_DIR = "/content/drive/MyDrive/CREMAD/Embeddings/ResNet18/val"
BATCH_SIZE = 8
SHOTS = 1024

# Read config from model folder to filter data
config_path = os.path.join(os.path.dirname(MODEL_PATH), "config.json")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config_data = json.load(f)
    selected_classes = config_data.get("selected_classes", ['ANG', 'SAD'])
else:
    selected_classes = ['ANG', 'SAD']

print(f"Loaded config from {config_path}")
print(f"Filtering dataset for classes: {selected_classes}")

# Make sure src modules can be found
sys.path.append(os.path.abspath("."))

from src.model_builder import build_model
from src.dataset import AudioFeatureDataset
from torch.utils.data import DataLoader
from torchvision import transforms


## 2. IBM Quantum Hardware Setup
We'll query the IBM provider for available compute backends.

In [ ]:
!pip install qiskit-ibm-provider qiskit-ibm-runtime pennylane-qiskit
from qiskit_ibm_provider import IBMProvider
import pennylane as qml

# Authenticate with the provider
try:
    provider = IBMProvider(token=IBM_TOKEN)
except Exception as e:
    # Fallback to save account if running for the first time
    IBMProvider.save_account(token=IBM_TOKEN, overwrite=True)
    provider = IBMProvider()

# List available REAL backends (filter out simulators)
available_backends = [b.name for b in provider.backends() if not b.configuration().simulator]

print("Available Real IBM Quantum Backends:")
for b in available_backends:
    print(f" - {b}")

if not available_backends:
    raise RuntimeError("\u274c No real hardware backends currently available to your account.")

### Select your Backend
Set `BACKEND_NAME` to one of the real hardware instances printed above.

In [ ]:
# Example: 'ibm_kyoto', 'ibm_osaka'
BACKEND_NAME = available_backends[0] if available_backends else None

print(f"Selected Backend: {BACKEND_NAME}")

# Initialize the PennyLane device pointing to the real IBM hardware
# We use 'qiskit.ibmq' device allowing PennyLane to route to IBM Provider
n_qubits = 4 # Adjust based on your model architecture
try:
    dev = qml.device('qiskit.ibmq', wires=n_qubits, backend=BACKEND_NAME, provider=provider, shots=SHOTS)
    print(f"\u2705 PennyLane device connected to {BACKEND_NAME}")
except Exception as e:
    raise RuntimeError(f"\u274c Failed to initialize Qiskit IBMQ device: {e}")

## 3. Model Loading & Data Prep
Ensure the model is loaded as an `eval()` evaluation state.

In [ ]:
device = torch.device("cpu") # Inference on CPU is fine since the QNode handles the heavy quantum lifting

# Reconstruct class_names based on sorted selected_classes
class_names = sorted(selected_classes)

# Using standard repo build_model logic
model_config = {
    'base_model': 'emb_resnet18', 
    'quantum': True,
    'n_qubits': n_qubits,
    'q_depth': 2,
    'max_layers': 10,
    'q_delta': 0.01,
}

# Override using what we found in config if available
if 'config_data' in locals():
    for k in model_config.keys():
        if k in config_data['config']:
            model_config[k] = config_data['config'][k]

print("Instantiating Model...")
model = build_model(model_config, class_names, dataloaders={}, device=device)

# LOAD WEIGHTS
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Could not find model at {MODEL_PATH}")

print(f"Loading weights from {MODEL_PATH}...")
state_dict = torch.load(MODEL_PATH, map_location=device)
if isinstance(state_dict, torch.nn.Module):
    model = state_dict
else:
    model.load_state_dict(state_dict)

model.to(device)
model.eval() # CRITICAL for inference!
print("✅ Model loaded and set to evaluation mode.")

# DATA PREPARATION (Filtering subset)
try:
    filepaths = []
    labels = []
    for i, cls_name in enumerate(class_names):
        cls_dir = os.path.join(TEST_DATA_DIR, cls_name)
        if os.path.exists(cls_dir):
            for file in os.listdir(cls_dir):
                filepaths.append(os.path.join(cls_dir, file))
                labels.append(i)
                
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    
    test_dataset = AudioFeatureDataset(filepaths, labels, base_model=model_config['base_model'])
    # AudioFeatureDataset takes filepaths, labels, base_model instead of generic directory 
    # since we filtered our dataset manually above!

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
except Exception as e:
    print(f"⚠️ Warning: Data load failed. Make sure {TEST_DATA_DIR} exists with valid samples.")
    print(e)
    test_loader = []


## 4. Hardware Inference Execution
We will pass a small batch of inputs to the model. The quantum layers will be offloaded to the IBM cloud queue. This process might take time depending on queue lengths.

In [ ]:
import time
import torch.nn.functional as F

if not test_loader:
    print("No test data found. Creating a random dummy tensor to prove hardware execution...")
    # Base resnet18 expects 3-channel 224x224 images
    dummy_inputs = torch.randn(BATCH_SIZE, 3, 224, 224)
    dummy_labels = torch.randint(0, len(class_names), (BATCH_SIZE,))
    test_loader = [(dummy_inputs, dummy_labels)]

print(f"Executing batch of {BATCH_SIZE} samples on {BACKEND_NAME}...")
start_time = time.time()

# CRITICAL: Pure inference without gradients
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        # --- FORWARD PASS ON HARDWARE ---
        logits = model(inputs)

        # Metrics
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        print("\n--- Inference Results ---")
        for i in range(len(inputs)):
            print(f"Sample {i+1}:")
            print(f"  Predicted Class: {class_names[preds[i]]}")
            print(f"  Confidence:      {probs[i][preds[i]]:.4f}")
            if labels is not None:
                print(f"  Actual Class:    {class_names[labels[i]]}")

        # We only run ONE batch to avoid excessive IBM queueing
        break

end_time = time.time()
print(f"\n\u2705 Total Hardware Execution Time: {end_time - start_time:.2f} seconds")

## Checklist
- [x] Environment configured (`.env`)
- [x] Correct physical backend selected
- [x] Weights loaded successfully (`eval()`)
- [x] Inference execution finished